# QSENSE

## Hamiltonian generation

In [ ]:
### prepare molecule integrals - H2O

mol='h2o'
basis='sto3g'
rdist = 1.0

#construct geometry manually
from src.utils_hamiltonian import get_h2o_geom, generate_hamiltonian_integrals

geo = get_h2o_geom(rdist)
generate_hamiltonian_integrals(mol, basis, geo, 6, 8, './data/', str(rdist), run_fci=True, fci_ncas=7, fci_nelec=10)

In [ ]:
### prepare molecule integrals - N2

mol='n2'
basis='sto3g'
rdist = 1.0

#construct geometry manually
from src.utils_hamiltonian import generate_hamiltonian_integrals, get_n2_geom

geo = get_n2_geom(1.0)
generate_hamiltonian_integrals(mol, basis, geo, 8, 10, './data/', str(rdist), run_fci=True, fci_ncas=10, fci_nelec=14)

## QSENSE Ansatz initialization and run

In [ ]:
### ansatz setup h2o

phy_int_file = './data/h2o_sto3g_6o8e_phys_spatial_1.0'
mp2_ampld_thrsh= 0.0
num_parallel_process = 1
actmo_start, actmo_end = 1, 6
rdist = 1.0
Ethrsh_select_ia = 0.0
Uopt_thrsh = 1e-5
text_initial_orb_rot = False
text_opt_orb = True
intmo_start, intmo_end = 1, 6

In [ ]:
### ansatz setup n2

phy_int_file = './data/n2_sto3g_8o10e_phys_spatial_1.0'
mp2_ampld_thrsh= 0.0
num_parallel_process = 1
actmo_start, actmo_end = 2, 9
rdist = 1.0
Ethrsh_select_ia = 0.0
Uopt_thrsh = 1e-5
text_initial_orb_rot = False
text_opt_orb = True
intmo_start, intmo_end = 2, 9

In [ ]:
### run qsense - VO and PT
from src.CSF_UCSF_GS_matchU_optorb_E0select_ia_pair_symCSF_analytic_expG import run_qsense

run_qsense(phy_int_file,
            mp2_ampld_thrsh,
            num_parallel_process,
            actmo_start,
            actmo_end,
            rdist,
            Ethrsh_select_ia,
            Uopt_thrsh,
            text_initial_orb_rot,
            text_opt_orb,
            intmo_start,
            intmo_end)

## Benchmarking circuits and measurement cost  

Dump files for systems in the manuscript is in `./data/h2o_data` and `./data/n2_data`. Change file references as needed.

In [ ]:
### benchmark circuits and measurement cost - VO

from src.measurement_new.utils_basic import (
    copy_hamiltonian
)
from src.measurement_new.utils_ferm import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from src.measurement_new.utils_states import (
    convert_TZ_format_to_sparse_format,
    convert_dense_format_to_sparse_format,
    tz_state_seniority_config,
    compress_state,
    decompress_state,
    create_composite_state
)
from src.measurement_new.utils_m1_seniority import (
    project_out_seniority_symmetries
)
from src.measurement_new.utils_m2_factorize import (
    expand_tensor_product,
    expand_tensor_product_for_incomplete_qubit_set,
    get_indices_mapping_2_wvn_vo,
    factorize_state,
    evaluate_fully_classical_factors,
    obtain_coarse_dicts,
    QC_assignment_from_qubit_labels
)
from src.measurement_new.utils_m3_swap import (
    XorY_augment
)
from src.measurement_new.utils_m4_partitioning import (
    sorted_insertion_decomposition
)
from src.measurement_new.utils_results import (
    variance_of_decomp,
    sampling_cost
)
from openfermion import (
    get_sparse_operator,
    jordan_wigner,
    count_qubits
)

import numpy as np
import pickle
import sys

from src.circuits.circuits_csf import get_csfs_from_dump
from src.circuits.utils_circuit import get_decomp_circuits_estimators, show_state, verify_circuit_state
from qiskit import QuantumCircuit, transpile
from src.circuits.circuits_swap import get_parallelswap_subcircuit

# load Q-SENSE basis states

molecule        = 'h2o'
bond_length     = '1.0'
filename        = './data/h2o_sto3g_6o8e_phys_spatial_1.0_UCSF_sym_comp.dump'#'./data/{molecule}_data/UCSF_sym_comp_for_Praveen_Smik_{bond_length}.dump'

with open(filename, 'rb') as f:
    (
    CSF_tz_states,
    W_amplitudes,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    UCSF_tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

CSFs = get_csfs_from_dump(filename, verify_states=True, verbose=True)

# rotate orbitals and obtain Hamiltonian operator

if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Hfer    = make_short_H_ferm_op(Enuc, obt, tbt)
Hqub    = jordan_wigner(Hfer)

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits

# process information so that we can taper and factorize the Q-SENSE states

Nstates          = len(UCSF_tz_states)
configs          = [tz_state_seniority_config(tz_state) for tz_state in UCSF_tz_states]
UCSF_information = [get_indices_mapping_2_wvn_vo(CSF_tz_states[i], W_amplitudes[i], Norb) for i in range(Nstates)]

SW_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'W']) for i in range(Nstates)]
SV_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'V']) for i in range(Nstates)]
SN_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'N']) for i in range(Nstates)]
state_type_list  = [UCSF_information[i][1] for i in range(Nstates)]

# taper and factorize the Q-SENSE basis states

statevectors                    = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in UCSF_tz_states]
tapered_statevectors            = [convert_dense_format_to_sparse_format(compress_state(psi.toarray()[0])) for psi in statevectors]
factorized_tapered_statevectors = [factorize_state(tapered_statevectors[i], SW_list[i], SV_list[i], SN_list[i], state_type_list[i]) 
                                   for i in range(Nstates)]


Hsub       = np.zeros([Nstates, Nstates], dtype=np.complex128)
sig_matrix = np.zeros([Nstates, Nstates], dtype=np.complex128)

def get_quantum_indices(bra_f, ket_f, bra_labels, ket_labels):
    quantum_indices = []

    join_partition, coarse_dict_bra, coarse_dict_ket = obtain_coarse_dicts(bra_f, ket_f)
    QC_assignment_dict                               = QC_assignment_from_qubit_labels(bra_labels, ket_labels, join_partition)

    for k, v in QC_assignment_dict.items():
        if v == 'Q':
            for idx in k:
                quantum_indices.append(idx)
    return quantum_indices

circuit_cx_counts = []
circuit_depth = []
tol=1e-5

for i in range(Nstates):
    
    print('CSF index: ', f'{i, i}')

    ket_f      = factorized_tapered_statevectors[i]
    ket_labels = UCSF_information[i][0]
    ket_config = configs[i]

    Htapered        = project_out_seniority_symmetries(Hqub, Nqubits, ket_config, ket_config)
    HQ, ketQ, _, NQ = evaluate_fully_classical_factors(ket_f, ket_f, ket_labels, ket_labels, Htapered)
    quantum_indices = get_quantum_indices(ket_f, ket_f, ket_labels, ket_labels)

    if NQ == 0 or np.sum(np.abs(list(HQ.terms.values()))) <=tol:
        Hsub[i,i] = HQ.constant

        circuit_cx_counts.append(0)
        circuit_depth.append(0)

    else:
        HQsparse   = get_sparse_operator(HQ)
        ketQ       = convert_dense_format_to_sparse_format(ketQ)
        Hsub[i,i] = (ketQ @ HQsparse @ ketQ.T)[0,0]

        HQ              -= HQ.constant
        HQ.compress()
        decomp = sorted_insertion_decomposition(HQ, 'fc')
        var_metric       = variance_of_decomp(decomp, ketQ, NQ, general=True)
        sig_matrix[i,i] = np.sqrt(var_metric)

        ### build circuits!
        decomp, meas_circuits, z_ops = get_decomp_circuits_estimators(HQ, NQ, methodtag='fc')
        csf_circ = CSFs[i].get_tapered_full_circuit(quantum_indices)

        assert verify_circuit_state(csf_circ, ketQ)
        csf_circ_t = transpile(csf_circ, basis_gates=['u3', 'cx'], optimization_level=3)
        
        circuit_cx_counts.append(csf_circ_t.num_nonlocal_gates())
        circuit_depth.append(csf_circ_t.depth())

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            
            print('CSF pair index: ', f'{i, j}')

            ij_shift           = 0.5 * (Hsub[i,i] + Hsub[j,j])

            bra_f              = factorized_tapered_statevectors[i]
            bra_labels         = UCSF_information[i][0]
            bra_config         = configs[i]

            ket_f              = factorized_tapered_statevectors[j]
            ket_labels         = UCSF_information[j][0]
            ket_config         = configs[j]

            Htapered           = project_out_seniority_symmetries(Hqub - ij_shift, Nqubits, bra_config, ket_config)
            HQ, braQ, ketQ, NQ = evaluate_fully_classical_factors(bra_f, ket_f, bra_labels, ket_labels, Htapered)
            quantum_qubits = get_quantum_indices(bra_f, ket_f, bra_labels, ket_labels)

            if NQ == 0 or np.sum(np.abs(list(HQ.terms.values()))) <=tol:
                Hsub[i,j] = HQ.constant
                Hsub[j,i] = HQ.constant

                #off-diagonal used twice
                circuit_cx_counts.append(0)
                circuit_depth.append(0)
                circuit_cx_counts.append(0)
                circuit_depth.append(0)

            else:
                HQsparse         = get_sparse_operator(HQ, NQ)
                braQ             = convert_dense_format_to_sparse_format(braQ)
                ketQ             = convert_dense_format_to_sparse_format(ketQ)
                Hsub[i,j]       = (braQ @ HQsparse @ ketQ.T)[0,0]
                Hsub[j,i]       = (braQ @ HQsparse @ ketQ.T)[0,0]

                comp             = create_composite_state(braQ, ketQ, NQ)
                HQ_aug           = XorY_augment(HQ, NQ)
                decomp           = sorted_insertion_decomposition(HQ_aug, 'fc')
                var_metric       = variance_of_decomp(decomp, comp, NQ + 1, general=True)
                sig_matrix[i,j] = np.sqrt(var_metric)
                sig_matrix[j,i] = np.sqrt(var_metric)

                ### build circuits!
                decomp, meas_circuits, z_ops = get_decomp_circuits_estimators(HQ_aug, NQ + 1, methodtag='fc')
                csf_circ = get_parallelswap_subcircuit(CSFs[i], CSFs[j], quantum_qubits=quantum_qubits, control_qubit_pos=NQ)
                #assert verify_circuit_state(csf_circ, comp, truncate_bitstrings=list(range(NQ+1)))
                csf_circ_t = transpile(csf_circ, basis_gates=['u3', 'cx'], optimization_level=3)
                
                #off-diagonal used twice
                circuit_cx_counts.append(csf_circ_t.num_nonlocal_gates())
                circuit_depth.append(csf_circ_t.depth())
                circuit_cx_counts.append(csf_circ_t.num_nonlocal_gates())
                circuit_depth.append(csf_circ_t.depth())

vals, vecs = np.linalg.eigh(Hsub)
Egs        = vals[0]
c          = vecs[:,0]
cost       = sampling_cost(c, sig_matrix)


print(f'''
    Final Results:
        Method              : {'VO'}
        Molecule            : {molecule}
        Bond Length         : {bond_length}
        Ground State Energy : {Egs}
        Sampling Cost       : {cost}
''')

import numpy as np
print('CX counts:\nMean: {}, Max: {}'.format(np.mean(circuit_cx_counts), np.max(circuit_cx_counts)))
print('Depth:\nMean: {}, Max: {}'.format(np.mean(circuit_depth), np.max(circuit_depth)))


In [ ]:
### benchmark circuits and measurement cost - PT

from src.measurement_new.utils_basic import (
    copy_hamiltonian
)
from src.measurement_new.utils_ferm import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from src.measurement_new.utils_states import (
    convert_TZ_format_to_sparse_format,
    convert_dense_format_to_sparse_format,
    tz_state_seniority_config,
    compress_state,
    decompress_state,
    create_composite_state
)
from src.measurement_new.utils_m1_seniority import (
    project_out_seniority_symmetries
)
from src.measurement_new.utils_m2_factorize import (
    expand_tensor_product,
    expand_tensor_product_for_incomplete_qubit_set,
    get_indices_mapping_2_wvn,
    factorize_state,
    evaluate_fully_classical_factors,
    obtain_coarse_dicts,
    QC_assignment_from_qubit_labels
)
from src.measurement_new.utils_m3_swap import (
    XorY_augment
)
from src.measurement_new.utils_m4_partitioning import (
    sorted_insertion_decomposition
)
from src.measurement_new.utils_results import (
    variance_of_decomp,
    sampling_cost
)
from openfermion import (
    get_sparse_operator,
    jordan_wigner,
    count_qubits
)

import numpy as np
import pickle
import sys

from src.circuits.circuits_csf import get_csfs_from_dump, get_Uext_csfs_from_dump
from src.circuits.utils_circuit import get_decomp_circuits_estimators, show_state, verify_circuit_state, get_sparse_state
from qiskit import QuantumCircuit, transpile
from src.circuits.circuits_swap import get_parallelswap_subcircuit
from scipy.sparse.linalg import norm

# load Q-SENSE basis states

molecule        = 'h2o'
bond_length     = '1.0'
filename        = './data/h2o_sto3g_6o8e_phys_spatial_1.0_Uext_CSF.dump'#f'./data/{molecule}_data/Uext_CSF_for_Praveen_Smik_{bond_length}.dump'

CSFs = get_Uext_csfs_from_dump(filename, verify_states=False, use_opt_amplitudes=False, verbose=True)

with open(filename, 'rb') as f:
    (
    list_list_refCSF,
    list_list_Uext_mp2_CSF,
    list_list_Uext_mp2_ampld,
    list_list_Uext_opt_ampld,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

# rotate orbitals and obtain Hamiltonian operator

if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Hfer    = make_short_H_ferm_op(Enuc, obt, tbt)
Hqub    = jordan_wigner(Hfer)

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits

# obtain relevant information about Q-SENSE states (UCSFs, CSFs, W information) in a linear list

UCSF_tz_states = []
CSF_tz_states  = []
W_amplitudes   = []

for i, ucsf_list in enumerate(list_list_Uext_mp2_CSF):
    for j, ucsf in enumerate(ucsf_list):
        UCSF_tz_states.append(ucsf)
        CSF_tz_states.append(list_list_refCSF[i][j])
        W_amplitudes.append(list_list_Uext_mp2_ampld[i])

# process information so that we can taper and factorize the Q-SENSE states

Nstates          = len(UCSF_tz_states)
configs          = [tz_state_seniority_config(tz_state) for tz_state in UCSF_tz_states]
UCSF_information = [get_indices_mapping_2_wvn(CSF_tz_states[i], W_amplitudes[i], Norb) for i in range(Nstates)]

SW_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'W']) for i in range(Nstates)]
SV_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'V']) for i in range(Nstates)]
SN_list          = [tuple([k for k, v in UCSF_information[i][0].items() if v == 'N']) for i in range(Nstates)]
state_type_list  = [UCSF_information[i][1] for i in range(Nstates)]

# taper and factorize the Q-SENSE basis states

statevectors                    = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in UCSF_tz_states]
tapered_statevectors            = [convert_dense_format_to_sparse_format(compress_state(psi.toarray()[0])) for psi in statevectors]
factorized_tapered_statevectors = [factorize_state(tapered_statevectors[i], SW_list[i], SV_list[i], SN_list[i], state_type_list[i]) 
                                   for i in range(Nstates)]


Hsub       = np.zeros([Nstates, Nstates], dtype=np.complex128)
sig_matrix = np.zeros([Nstates, Nstates], dtype=np.complex128)

def get_quantum_indices(bra_f, ket_f, bra_labels, ket_labels):
    quantum_indices = []

    join_partition, coarse_dict_bra, coarse_dict_ket = obtain_coarse_dicts(bra_f, ket_f)
    QC_assignment_dict                               = QC_assignment_from_qubit_labels(bra_labels, ket_labels, join_partition)

    for k, v in QC_assignment_dict.items():
        if v == 'Q':
            for idx in k:
                quantum_indices.append(idx)
    return quantum_indices

circuit_cx_counts = []
circuit_depth = []
tol=1e-5

for i in range(Nstates):
    
    print('CSF index: ', f'{i, i}')

    ket_f      = factorized_tapered_statevectors[i]
    ket_labels = UCSF_information[i][0]
    ket_config = configs[i]

    Htapered        = project_out_seniority_symmetries(Hqub, Nqubits, ket_config, ket_config)
    HQ, ketQ, _, NQ = evaluate_fully_classical_factors(ket_f, ket_f, ket_labels, ket_labels, Htapered)
    quantum_indices = get_quantum_indices(ket_f, ket_f, ket_labels, ket_labels)

    if NQ == 0 or np.sum(np.abs(list(HQ.terms.values()))) <=tol:
        Hsub[i,i] = HQ.constant

        circuit_cx_counts.append(0)
        circuit_depth.append(0)

    else:
        HQsparse   = get_sparse_operator(HQ)
        ketQ       = convert_dense_format_to_sparse_format(ketQ)
        Hsub[i,i] = (ketQ @ HQsparse @ ketQ.T)[0,0]

        HQ              -= HQ.constant
        HQ.compress()
        decomp = sorted_insertion_decomposition(HQ, 'fc')
        var_metric       = variance_of_decomp(decomp, ketQ, NQ, general=True)
        sig_matrix[i,i] = np.sqrt(var_metric)

        ### build circuits!
        decomp, meas_circuits, z_ops = get_decomp_circuits_estimators(HQ, NQ, methodtag='fc')
        csf_circ = CSFs[i].get_tapered_full_circuit(quantum_indices)

        assert verify_circuit_state(csf_circ, ketQ)
        csf_circ_t = transpile(csf_circ, basis_gates=['u3', 'cx'], optimization_level=3)
        
        circuit_cx_counts.append(csf_circ_t.num_nonlocal_gates())
        circuit_depth.append(csf_circ_t.depth())

        #circuits = [QuantumCircuit.compose(csf_circ, m) for m in meas_circuits]

        #verify state with ketQ

        

        # fragment_circuits[(i, i)] = circuits
        # fragment_z_ops[(i, i)] = z_ops
        # fragment_SDs[(i, i)] = frag_SD_of_decomp(decomp, ket_t, Nqubits//2, general=True)

for i in range(Nstates):
    for j in range(Nstates):
        if i > j:
            
            print('CSF pair index: ', f'{i, j}')

            ij_shift           = 0.5 * (Hsub[i,i] + Hsub[j,j])

            bra_f              = factorized_tapered_statevectors[i]
            bra_labels         = UCSF_information[i][0]
            bra_config         = configs[i]

            ket_f              = factorized_tapered_statevectors[j]
            ket_labels         = UCSF_information[j][0]
            ket_config         = configs[j]

            Htapered           = project_out_seniority_symmetries(Hqub - ij_shift, Nqubits, bra_config, ket_config)
            HQ, braQ, ketQ, NQ = evaluate_fully_classical_factors(bra_f, ket_f, bra_labels, ket_labels, Htapered)
            quantum_qubits = get_quantum_indices(bra_f, ket_f, bra_labels, ket_labels)

            if NQ == 0 or np.sum(np.abs(list(HQ.terms.values()))) <=tol:
                Hsub[i,j] = HQ.constant
                Hsub[j,i] = HQ.constant

                #off-diagonal used twice
                circuit_cx_counts.append(0)
                circuit_depth.append(0)
                circuit_cx_counts.append(0)
                circuit_depth.append(0)

            else:
                HQsparse         = get_sparse_operator(HQ, NQ)
                braQ             = convert_dense_format_to_sparse_format(braQ)
                ketQ             = convert_dense_format_to_sparse_format(ketQ)
                Hsub[i,j]       = (braQ @ HQsparse @ ketQ.T)[0,0]
                Hsub[j,i]       = (braQ @ HQsparse @ ketQ.T)[0,0]

                comp             = create_composite_state(braQ, ketQ, NQ)
                HQ_aug           = XorY_augment(HQ, NQ)
                decomp           = sorted_insertion_decomposition(HQ_aug, 'fc')
                var_metric       = variance_of_decomp(decomp, comp, NQ + 1, general=True)
                sig_matrix[i,j] = np.sqrt(var_metric)
                sig_matrix[j,i] = np.sqrt(var_metric)

                ### build circuits!
                decomp, meas_circuits, z_ops = get_decomp_circuits_estimators(HQ_aug, NQ + 1, methodtag='fc')
                csf_circ = get_parallelswap_subcircuit(CSFs[i], CSFs[j], quantum_qubits=quantum_qubits, control_qubit_pos=NQ)
                if not verify_circuit_state(csf_circ, comp, truncate_bitstrings=list(range(NQ+1))):
                    #for now check difference in absolute vectors
                    state = show_state(csf_circ)
                    bit_strings, coeffs = list(state.keys()), list(state.values())
                    truncate_bitstrings = list(range(NQ+1))

                    if truncate_bitstrings is not None:
                        for i in range(len(bit_strings)):
                            bs_new = ''
                            for j in truncate_bitstrings:
                                bs_new += bit_strings[i][j]
                            bit_strings[i] = bs_new 
                    
                    sparse_state = get_sparse_state({k: abs(v) for k, v in zip(bit_strings, coeffs)})

                    assert norm(abs(comp) - sparse_state) <= tol
                    
                csf_circ_t = transpile(csf_circ, basis_gates=['u3', 'cx'], optimization_level=3)
                
                #off-diagonal used twice
                circuit_cx_counts.append(csf_circ_t.num_nonlocal_gates())
                circuit_depth.append(csf_circ_t.depth())
                circuit_cx_counts.append(csf_circ_t.num_nonlocal_gates())
                circuit_depth.append(csf_circ_t.depth())

vals, vecs = np.linalg.eigh(Hsub)
Egs        = vals[0]
c          = vecs[:,0]
cost       = sampling_cost(c, sig_matrix)


print(f'''
    Final Results:
        Method              : {'PT'}
        Molecule            : {molecule}
        Bond Length         : {bond_length}
        Ground State Energy : {Egs}
        Sampling Cost       : {cost}
''')

import numpy as np
print('CX counts:\nMean: {}, Max: {}'.format(np.mean(circuit_cx_counts), np.max(circuit_cx_counts)))
print('Depth:\nMean: {}, Max: {}'.format(np.mean(circuit_depth), np.max(circuit_depth)))

output_filename = f'output_benchmark_n2'

with open(output_filename, 'a') as f:
    print('CX counts:\nMean: {}, Max: {}'.format(np.mean(circuit_cx_counts), np.max(circuit_cx_counts)), file=f)
    print('Depth:\nMean: {}, Max: {}'.format(np.mean(circuit_depth), np.max(circuit_depth)), file=f)